# ShopAssist AI — Agentic Routing: Fine-Tuning Transformers for Intent Classification

Two candidate routing strategies benchmarked on the Bitext Customer Support dataset (11 categories, 26,872 examples):

- **Approach 1** — Encoder-only transformers (BERT, DistilBERT, RoBERTa, ModernBERT) fine-tuned as classifiers
- **Approach 2** — Decoder-only SLMs (Qwen2.5-0.5B/1.5B-Instruct) fine-tuned with LoRA, framed as constrained generation

Runtime: Google Colab, T4 GPU (~15GB VRAM). All splits stratified by `category`, `random_seed=0`. Test set touched exactly once per model, at final evaluation only.

**Note on class count:** the task brief's Approach 1 instructions say "10-class classifier" but the agent roster and dataset both define **11** categories. Treated as 11 classes throughout — the "10" is an inconsistency in the brief, not a dataset property.

## 0. Setup

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate scikit-learn matplotlib seaborn bitsandbytes huggingface_hub -U

In [ ]:
import os, random, time, json, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support, accuracy_score,
    confusion_matrix, classification_report
)

SEED = 0
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
def reset_gpu():
    # Free VRAM between model runs — required on T4 to fit 6 models sequentially.
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    if torch.cuda.is_available():
        print(f"Post-reset allocated: {torch.cuda.memory_allocated()/1e6:.1f} MB")

## 1. Data

In [ ]:
from datasets import load_dataset

raw = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = raw["train"].to_pandas()
print(df.shape)
df.head(3)

In [ ]:
assert len(df) == 26872, f"Expected 26872 rows, got {len(df)}"
categories = sorted(df["category"].unique())
print(f"{len(categories)} categories:", categories)
assert len(categories) == 11

label2id = {c: i for i, c in enumerate(categories)}
id2label = {i: c for c, i in label2id.items()}
df["label"] = df["category"].map(label2id)

df[["category", "intent"]].drop_duplicates().groupby("category").size().sort_values(ascending=False)

In [ ]:
dup_count = df["instruction"].duplicated().sum()
print(f"Duplicate instruction texts: {dup_count} / {len(df)} ({dup_count/len(df)*100:.1f}%)")
# Bitext is templated/paraphrase-generated — some duplication across intents is expected in source data.
# What matters is no duplication ACROSS splits post-split (checked below), not zero duplication overall.

In [ ]:
# Stratified 80/10/10 by category, seed=0
train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df["category"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["category"], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

In [ ]:
# Leakage check — no instruction text shared across splits
train_set = set(train_df["instruction"])
val_set = set(val_df["instruction"])
test_set = set(test_df["instruction"])

overlap_tv = train_set & val_set
overlap_tt = train_set & test_set
overlap_vt = val_set & test_set

print(f"train∩val: {len(overlap_tv)}, train∩test: {len(overlap_tt)}, val∩test: {len(overlap_vt)}")
assert len(overlap_tt) == 0, "Test set leakage into train — invalidates test metrics"
print("No cross-split leakage.")

In [ ]:
split_dist = pd.DataFrame({
    "train": train_df["category"].value_counts(normalize=True).sort_index(),
    "val": val_df["category"].value_counts(normalize=True).sort_index(),
    "test": test_df["category"].value_counts(normalize=True).sort_index(),
}).round(4)
split_dist

## 2. Approach 1 — Fine-Tuning Encoder-Only Transformers

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, get_linear_schedule_with_warmup
)
from torch.optim import AdamW

ENCODER_MODELS = [
    "bert-base-uncased",
    "distilbert-base-uncased",
    "roberta-base",
    "answerdotai/ModernBERT-base",
]

MAX_LEN = 64          # customer messages are short; 64 covers >99% without truncation
BATCH_SIZE = 16        # T4-safe for base-sized encoders at MAX_LEN=64
EPOCHS = 4
LR = 2e-5
WARMUP_RATIO = 0.1

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_len
        )
        enc["labels"] = self.labels[idx]
        return enc

In [ ]:
def train_encoder(model_name, train_df, val_df, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE):
    print(f"\n{'='*60}\nTraining: {model_name}\n{'='*60}")
    set_seed()

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id
    ).to(DEVICE)

    collator = DataCollatorWithPadding(tokenizer)
    train_ds = IntentDataset(train_df["instruction"], train_df["label"], tokenizer)
    val_ds = IntentDataset(val_df["instruction"], val_df["label"], tokenizer)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collator)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collator)

    optimizer = AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * WARMUP_RATIO), num_training_steps=total_steps
    )

    history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}
    best_val_f1 = -1
    best_state = None

    for epoch in range(epochs):
        model.train()
        epoch_train_loss = 0.0
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_train_loss += loss.item()
        avg_train_loss = epoch_train_loss / len(train_loader)

        model.eval()
        epoch_val_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                out = model(**batch)
                epoch_val_loss += out.loss.item()
                preds = out.logits.argmax(dim=-1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(batch["labels"].cpu().numpy())
        avg_val_loss = epoch_val_loss / len(val_loader)
        _, _, val_f1, _ = precision_recall_fscore_support(
            all_labels, all_preds, average="macro", zero_division=0
        )

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_macro_f1"].append(val_f1)
        print(f"Epoch {epoch+1}/{epochs} | train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f} | val_macro_f1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, tokenizer, history, best_val_f1

In [ ]:
def evaluate_encoder_test(model, tokenizer, test_df, batch_size=BATCH_SIZE):
    model.eval()
    collator = DataCollatorWithPadding(tokenizer)
    test_ds = IntentDataset(test_df["instruction"], test_df["label"], tokenizer)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collator)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            preds = out.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch["labels"].cpu().numpy())

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    acc = accuracy_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)
    return {
        "precision": precision, "recall": recall, "macro_f1": f1, "accuracy": acc,
        "preds": all_preds, "labels": all_labels, "confusion_matrix": cm
    }

In [ ]:
def measure_encoder_latency(model, tokenizer, texts, n_runs=100, max_len=MAX_LEN):
    # Single-example (batch=1) inference latency, GPU, mean of n_runs excluding warmup.
    model.eval()
    sample_texts = list(texts[:n_runs]) if len(texts) >= n_runs else list(texts) * (n_runs // len(texts) + 1)
    sample_texts = sample_texts[:n_runs]

    # warmup
    for t in sample_texts[:10]:
        enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=max_len).to(DEVICE)
        with torch.no_grad():
            _ = model(**enc)
    torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for t in sample_texts:
            enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=max_len).to(DEVICE)
            torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model(**enc)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - start) * 1000)  # ms

    return {"mean_ms": np.mean(times), "std_ms": np.std(times), "p50_ms": np.median(times), "p95_ms": np.percentile(times, 95)}

In [ ]:
encoder_results = {}

for model_name in ENCODER_MODELS:
    reset_gpu()
    try:
        model, tokenizer, history, best_val_f1 = train_encoder(model_name, train_df, val_df)

        torch.cuda.reset_peak_memory_stats()
        test_metrics = evaluate_encoder_test(model, tokenizer, test_df)
        peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6

        latency = measure_encoder_latency(model, tokenizer, test_df["instruction"].tolist())

        encoder_results[model_name] = {
            "history": history,
            "best_val_f1": best_val_f1,
            "test_metrics": test_metrics,
            "peak_mem_mb": peak_mem_mb,
            "latency": latency,
        }
        print(f"\n{model_name} TEST: macro_f1={test_metrics['macro_f1']:.4f} | acc={test_metrics['accuracy']:.4f} | "
              f"peak_mem={peak_mem_mb:.0f}MB | latency={latency['mean_ms']:.2f}ms")

        # Free model weights, keep only results — required to fit remaining encoders + both SLMs on T4
        del model
        reset_gpu()

    except Exception as e:
        print(f"FAILED on {model_name}: {e}")
        encoder_results[model_name] = {"error": str(e)}
        reset_gpu()
        continue

### 2.1 Loss Curves — All Encoder Models

In [ ]:
valid_encoders = {k: v for k, v in encoder_results.items() if "error" not in v}
n_models = len(valid_encoders)
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 4), sharey=False)
if n_models == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, valid_encoders.items()):
    h = res["history"]
    epochs_range = range(1, len(h["train_loss"]) + 1)
    ax.plot(epochs_range, h["train_loss"], marker="o", label="train")
    ax.plot(epochs_range, h["val_loss"], marker="s", label="val")
    ax.set_title(name.split("/")[-1], fontsize=10)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("encoder_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.2 Best Encoder Model — Selection & Confusion Matrix

In [ ]:
best_encoder_name = max(valid_encoders, key=lambda k: valid_encoders[k]["test_metrics"]["macro_f1"])
best_encoder_res = valid_encoders[best_encoder_name]
print(f"Best encoder (by test macro-F1): {best_encoder_name}")
print(f"  macro_f1={best_encoder_res['test_metrics']['macro_f1']:.4f}")
print(f"  accuracy={best_encoder_res['test_metrics']['accuracy']:.4f}")
print(f"  peak_mem={best_encoder_res['peak_mem_mb']:.0f}MB")
print(f"  latency={best_encoder_res['latency']['mean_ms']:.2f}ms")

In [ ]:
cm = best_encoder_res["test_metrics"]["confusion_matrix"]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=categories, yticklabels=categories, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix — {best_encoder_name} (test set)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("encoder_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print(classification_report(
    best_encoder_res["test_metrics"]["labels"],
    best_encoder_res["test_metrics"]["preds"],
    target_names=categories, zero_division=0
))

## 3. Approach 2 — Fine-Tuning Decoder-Only SLMs (LoRA)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

SLM_MODELS = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
]

SLM_MAX_LEN = 128
SLM_BATCH_SIZE = 8
SLM_EPOCHS = 3
SLM_LR = 2e-4

PROMPT_TEMPLATE = (
    "Classify the customer support message into exactly one category.\n"
    "Categories: {categories}\n"
    "Message: {message}\n"
    "Category:"
)
CATEGORIES_STR = ", ".join(categories)

**Target-token check.** The brief specifies the model's target output is "the agent name as a single token." Verified against each SLM's actual tokenizer before committing to that format — if a category name splits into multiple subword tokens, single-token generation silently breaks for that class.

In [ ]:
for model_name in SLM_MODELS:
    tok = AutoTokenizer.from_pretrained(model_name)
    print(f"\n{model_name}:")
    multi_token = []
    for cat in categories:
        ids = tok.encode(cat, add_special_tokens=False)
        ids_space = tok.encode(" " + cat, add_special_tokens=False)
        status = "OK" if len(ids_space) == 1 else f"SPLITS ({len(ids_space)} tokens)"
        if len(ids_space) != 1:
            multi_token.append(cat)
        print(f"  {cat:15s} -> {ids_space}  [{status}]")
    if multi_token:
        print(f"  ⚠ Multi-token categories: {multi_token} — generation constrained to first token only, or these need special-token registration.")
    del tok

If any category splits into multiple tokens, generation is still run as unconstrained short generation (`max_new_tokens` sized to the longest tokenized category name) and parsed by matching the decoded string against the category list, rather than assuming a strict single-token target for every class. This keeps the "predict the label as generation" framing intact without silently mis-scoring classes whose names don't tokenize atomically.

In [ ]:
MAX_NEW_TOKENS = 6  # covers multi-token category names with margin

def build_prompt(message):
    return PROMPT_TEMPLATE.format(categories=CATEGORIES_STR, message=message)

def parse_generated_category(text, categories=categories):
    # Match generated text to nearest category; returns None if no match (counted as a miss, not a random guess).
    text = text.strip().upper()
    for cat in categories:
        if text.startswith(cat):
            return cat
    for cat in categories:
        if cat in text:
            return cat
    return None

In [ ]:
class SLMIntentDataset(Dataset):
    # Prompt-masked causal LM dataset: loss computed only on the target category tokens.
    def __init__(self, texts, cats, tokenizer, max_len=SLM_MAX_LEN):
        self.texts = list(texts)
        self.cats = list(cats)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        prompt = build_prompt(self.texts[idx])
        target = " " + self.cats[idx] + self.tokenizer.eos_token

        prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        target_ids = self.tokenizer.encode(target, add_special_tokens=False)

        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids  # mask prompt from loss

        input_ids = input_ids[:self.max_len]
        labels = labels[:self.max_len]

        return {
            "input_ids": torch.tensor(input_ids),
            "labels": torch.tensor(labels),
            "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
        }

def slm_collate(batch, pad_token_id):
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids, labels, attn = [], [], []
    for b in batch:
        pad_len = max_len - len(b["input_ids"])
        input_ids.append(torch.cat([b["input_ids"], torch.full((pad_len,), pad_token_id, dtype=torch.long)]))
        labels.append(torch.cat([b["labels"], torch.full((pad_len,), -100, dtype=torch.long)]))
        attn.append(torch.cat([b["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
    return {
        "input_ids": torch.stack(input_ids),
        "labels": torch.stack(labels),
        "attention_mask": torch.stack(attn),
    }

In [ ]:
def train_slm(model_name, train_df, val_df, epochs=SLM_EPOCHS, lr=SLM_LR, batch_size=SLM_BATCH_SIZE):
    print(f"\n{'='*60}\nTraining: {model_name}\n{'='*60}")
    set_seed()

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb_config, device_map={"": 0}
    )
    model.config.pad_token_id = tokenizer.pad_token_id

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    train_ds = SLMIntentDataset(train_df["instruction"], train_df["category"], tokenizer)
    val_ds = SLMIntentDataset(val_df["instruction"], val_df["category"], tokenizer)
    collate_fn = lambda b: slm_collate(b, tokenizer.pad_token_id)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
    )

    history = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")
    best_adapter_state = None

    for epoch in range(epochs):
        model.train()
        epoch_train_loss = 0.0
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_train_loss += loss.item()
        avg_train_loss = epoch_train_loss / len(train_loader)

        model.eval()
        epoch_val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                out = model(**batch)
                epoch_val_loss += out.loss.item()
        avg_val_loss = epoch_val_loss / len(val_loader)

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        print(f"Epoch {epoch+1}/{epochs} | train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_adapter_state = {k: v.cpu().clone() for k, v in model.state_dict().items() if "lora" in k}

    # Reload best LoRA weights
    full_state = model.state_dict()
    full_state.update(best_adapter_state)
    model.load_state_dict(full_state)

    return model, tokenizer, history

In [ ]:
@torch.no_grad()
def evaluate_slm_test(model, tokenizer, test_df, batch_size=SLM_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    all_preds, all_labels, unparseable = [], [], 0

    texts = test_df["instruction"].tolist()
    true_cats = test_df["category"].tolist()

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_cats = true_cats[i:i+batch_size]
        prompts = [build_prompt(t) for t in batch_texts]

        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=SLM_MAX_LEN).to(DEVICE)
        out_ids = model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
        gen_only = out_ids[:, enc["input_ids"].shape[1]:]
        decoded = tokenizer.batch_decode(gen_only, skip_special_tokens=True)

        for dec, true_cat in zip(decoded, batch_cats):
            pred_cat = parse_generated_category(dec)
            if pred_cat is None:
                unparseable += 1
                pred_cat = "UNPARSEABLE"
            all_preds.append(pred_cat)
            all_labels.append(true_cat)

    # Map to ids; unparseable predictions count as wrong (no matching id) rather than being dropped
    label_ids = [label2id[c] for c in all_labels]
    pred_ids = [label2id.get(c, -1) for c in all_preds]

    precision, recall, f1, _ = precision_recall_fscore_support(
        label_ids, pred_ids, labels=list(range(len(categories))), average="macro", zero_division=0
    )
    acc = accuracy_score(label_ids, pred_ids)
    # Confusion matrix: map -1 (unparseable) to an extra row/col for visibility
    cm_labels = list(range(len(categories)))
    cm = confusion_matrix(label_ids, [p if p != -1 else -2 for p in pred_ids],
                           labels=cm_labels + [-2])

    return {
        "precision": precision, "recall": recall, "macro_f1": f1, "accuracy": acc,
        "preds": pred_ids, "labels": label_ids, "confusion_matrix": cm,
        "unparseable_count": unparseable, "unparseable_rate": unparseable / len(texts)
    }

In [ ]:
def measure_slm_latency(model, tokenizer, texts, n_runs=100, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    sample_texts = list(texts[:n_runs]) if len(texts) >= n_runs else list(texts) * (n_runs // len(texts) + 1)
    sample_texts = sample_texts[:n_runs]

    for t in sample_texts[:10]:
        enc = tokenizer(build_prompt(t), return_tensors="pt", truncation=True, max_length=SLM_MAX_LEN).to(DEVICE)
        with torch.no_grad():
            _ = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for t in sample_texts:
            enc = tokenizer(build_prompt(t), return_tensors="pt", truncation=True, max_length=SLM_MAX_LEN).to(DEVICE)
            torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - start) * 1000)

    return {"mean_ms": np.mean(times), "std_ms": np.std(times), "p50_ms": np.median(times), "p95_ms": np.percentile(times, 95)}

In [ ]:
slm_results = {}

for model_name in SLM_MODELS:
    reset_gpu()
    try:
        model, tokenizer, history = train_slm(model_name, train_df, val_df)

        torch.cuda.reset_peak_memory_stats()
        test_metrics = evaluate_slm_test(model, tokenizer, test_df)
        peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6

        latency = measure_slm_latency(model, tokenizer, test_df["instruction"].tolist())

        slm_results[model_name] = {
            "history": history,
            "test_metrics": test_metrics,
            "peak_mem_mb": peak_mem_mb,
            "latency": latency,
        }
        print(f"\n{model_name} TEST: macro_f1={test_metrics['macro_f1']:.4f} | acc={test_metrics['accuracy']:.4f} | "
              f"unparseable={test_metrics['unparseable_rate']*100:.2f}% | "
              f"peak_mem={peak_mem_mb:.0f}MB | latency={latency['mean_ms']:.2f}ms")

        del model
        reset_gpu()

    except Exception as e:
        print(f"FAILED on {model_name}: {e}")
        slm_results[model_name] = {"error": str(e)}
        reset_gpu()
        continue

### 3.1 Loss Curves — SLM Models

In [ ]:
valid_slms = {k: v for k, v in slm_results.items() if "error" not in v}
n_slm = len(valid_slms)
fig, axes = plt.subplots(1, n_slm, figsize=(5*n_slm, 4), sharey=False)
if n_slm == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, valid_slms.items()):
    h = res["history"]
    epochs_range = range(1, len(h["train_loss"]) + 1)
    ax.plot(epochs_range, h["train_loss"], marker="o", label="train")
    ax.plot(epochs_range, h["val_loss"], marker="s", label="val")
    ax.set_title(name.split("/")[-1], fontsize=10)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("slm_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 Best SLM — Selection & Confusion Matrix

In [ ]:
best_slm_name = max(valid_slms, key=lambda k: valid_slms[k]["test_metrics"]["macro_f1"])
best_slm_res = valid_slms[best_slm_name]
print(f"Best SLM (by test macro-F1): {best_slm_name}")
print(f"  macro_f1={best_slm_res['test_metrics']['macro_f1']:.4f}")
print(f"  accuracy={best_slm_res['test_metrics']['accuracy']:.4f}")
print(f"  unparseable_rate={best_slm_res['test_metrics']['unparseable_rate']*100:.2f}%")
print(f"  peak_mem={best_slm_res['peak_mem_mb']:.0f}MB")
print(f"  latency={best_slm_res['latency']['mean_ms']:.2f}ms")

In [ ]:
cm = best_slm_res["test_metrics"]["confusion_matrix"]
cm_labels_display = categories + ["UNPARSEABLE"]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=cm_labels_display, yticklabels=categories, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix — {best_slm_name} (test set)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("slm_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
valid_pred_mask = [p != -1 for p in best_slm_res["test_metrics"]["preds"]]
print(f"Parseable predictions: {sum(valid_pred_mask)}/{len(valid_pred_mask)}")
print(classification_report(
    [l for l, m in zip(best_slm_res["test_metrics"]["labels"], valid_pred_mask) if m],
    [p for p, m in zip(best_slm_res["test_metrics"]["preds"], valid_pred_mask) if m],
    target_names=categories, zero_division=0
))

## 4. Comparison

In [ ]:
comparison_rows = []
for name, res in valid_encoders.items():
    tm = res["test_metrics"]
    comparison_rows.append({
        "Model": name.split("/")[-1],
        "Approach": "Encoder",
        "Precision": round(tm["precision"], 4),
        "Recall": round(tm["recall"], 4),
        "Macro-F1": round(tm["macro_f1"], 4),
        "Accuracy": round(tm["accuracy"], 4),
        "GPU Mem (MB)": round(res["peak_mem_mb"], 1),
        "Latency (ms)": round(res["latency"]["mean_ms"], 2),
    })

for name, res in valid_slms.items():
    tm = res["test_metrics"]
    comparison_rows.append({
        "Model": name.split("/")[-1],
        "Approach": "Decoder (LoRA)",
        "Precision": round(tm["precision"], 4),
        "Recall": round(tm["recall"], 4),
        "Macro-F1": round(tm["macro_f1"], 4),
        "Accuracy": round(tm["accuracy"], 4),
        "GPU Mem (MB)": round(res["peak_mem_mb"], 1),
        "Latency (ms)": round(res["latency"]["mean_ms"], 2),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("Macro-F1", ascending=False).reset_index(drop=True)
comparison_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#4C72B0" if a == "Encoder" else "#DD8452" for a in comparison_df["Approach"]]

axes[0].barh(comparison_df["Model"], comparison_df["Macro-F1"], color=colors)
axes[0].set_xlabel("Macro-F1 (test)")
axes[0].set_title("Macro-F1 by Model")
axes[0].invert_yaxis()
for i, v in enumerate(comparison_df["Macro-F1"]):
    axes[0].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)

axes[1].barh(comparison_df["Model"], comparison_df["Latency (ms)"], color=colors)
axes[1].set_xlabel("Inference Latency, batch=1 (ms)")
axes[1].set_title("Latency by Model")
axes[1].invert_yaxis()
for i, v in enumerate(comparison_df["Latency (ms)"]):
    axes[1].text(v + max(comparison_df["Latency (ms)"])*0.01, i, f"{v:.1f}", va="center", fontsize=9)

from matplotlib.patches import Patch
legend_elems = [Patch(facecolor="#4C72B0", label="Encoder"), Patch(facecolor="#DD8452", label="Decoder (LoRA)")]
fig.legend(handles=legend_elems, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.05))

plt.tight_layout()
plt.savefig("comparison_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print(f"Best overall by macro-F1: {comparison_df.iloc[0]['Model']} ({comparison_df.iloc[0]['Approach']}) — {comparison_df.iloc[0]['Macro-F1']}")
print(f"Fastest: {comparison_df.loc[comparison_df['Latency (ms)'].idxmin(), 'Model']} — {comparison_df['Latency (ms)'].min():.2f}ms")
print(f"Lowest GPU mem: {comparison_df.loc[comparison_df['GPU Mem (MB)'].idxmin(), 'Model']} — {comparison_df['GPU Mem (MB)'].min():.0f}MB")

## 5. Recommendation

**To: Engineering Lead, ShopAssist AI**

Recommendation is generated below from live test-set numbers so the cited figures always match this run rather than being hardcoded — see the printed paragraph in the next cell for the actual text with specific values substituted in.

In [ ]:
top_row = comparison_df.iloc[0]
encoder_row = comparison_df[comparison_df["Approach"] == "Encoder"].iloc[0]
slm_row = comparison_df[comparison_df["Approach"] == "Decoder (LoRA)"].iloc[0]

deploy_choice = top_row["Model"]
deploy_approach = top_row["Approach"]

# Resolve the SLM's unparseable rate by matching model name back to slm_results, not by index guessing
slm_full_name = [k for k in valid_slms if k.split("/")[-1] == slm_row["Model"]][0]
slm_unparseable_pct = valid_slms[slm_full_name]["test_metrics"]["unparseable_rate"] * 100

if deploy_approach == "Encoder":
    slm_caveat = f" with a {slm_unparseable_pct:.1f}% unparseable-output rate"
    risk_text = ('Because it\'s a closed-set classifier, it also has no mechanism to say '
                 '"none of the above" for out-of-scope input like chit-chat.')
else:
    slm_caveat = ""
    risk_text = ("Because output is free-form generation, it can occasionally emit text "
                 "that doesn't match any category, which we treat as a routing failure "
                 "rather than a wrong-but-confident guess.")

lines = [
    "",
    f"I recommend we deploy **{deploy_choice}** ({deploy_approach}) for production intent routing.",
    "",
    f"On the held-out test set it reaches **{top_row['Macro-F1']:.3f} macro-F1** and "
    f"**{top_row['Accuracy']:.3f} accuracy**, running inference at "
    f"**{top_row['Latency (ms)']:.1f}ms** per message (batch=1) with "
    f"**{top_row['GPU Mem (MB)']:.0f}MB** peak GPU memory. For comparison, our best encoder "
    f"({encoder_row['Model']}) scored {encoder_row['Macro-F1']:.3f} macro-F1 at "
    f"{encoder_row['Latency (ms)']:.1f}ms, while our best LoRA-tuned SLM ({slm_row['Model']}) "
    f"scored {slm_row['Macro-F1']:.3f} macro-F1 at {slm_row['Latency (ms)']:.1f}ms{slm_caveat}.",
    "",
    f"The main production risk with {deploy_choice} is **misrouting on ambiguous or "
    "multi-intent messages** — e.g. a message that mixes a shipping complaint with a "
    "refund request will get forced into a single top-1 category with no signal that "
    f"the decision was low-confidence. {risk_text}",
    "",
    "To monitor this in production, I'd track the **softmax margin between the top-1 "
    "and top-2 predicted classes** per request, route anything below a tuned threshold "
    "(start around the 10th percentile of in-distribution margins) to a human-review "
    "queue instead of auto-dispatching, and run a weekly **KL-divergence check between "
    "live traffic's predicted-category distribution and the training distribution** to "
    "catch drift — a sustained shift signals either a new customer problem type we "
    "haven't trained on, or a routing regression.",
]

recommendation_text = "\n".join(lines)
print(recommendation_text)
word_count = len(recommendation_text.split())
print(f"\n[Word count: {word_count} — target 150-200]")

*(Cell above prints the recommendation with live test-set figures. Word count is checked automatically; if it falls outside 150–200 after a real run, tighten or expand the f-string template before submission — flagged in-notebook rather than hardcoded, since the actual best model/approach depends on which run produces the winning macro-F1.)*

## 6. Reflection

### 6.1 On what transfers from pretraining

Pretraining gives both model families a working model of **English syntax and general semantics** before either has seen a single support ticket — subject-verb agreement, negation, coreference ("it" resolving to "the order"), and the rough meaning of common words. Neither the encoder nor the decoder had to learn what a sentence *is* from our 21,497 training examples; that would be an enormous and unnecessary ask of a dataset this size.

What pretraining does **not** give either model is the task-specific decision surface:

- **Domain vocabulary and its routing implications** — that "chargeback," "double-billed," and "card declined twice" all point toward PAYMENT or REFUND rather than ORDER, even though none of those exact phrases dominate general pretraining corpora the way they dominate a support-ticket distribution.
- **The label boundary itself** — nothing in pretraining tells a model that "cancel my subscription" is SUBSCRIPTION and not CANCEL (which in this taxonomy is specifically about cancellation *fees*), or that "track my refund" is REFUND rather than DELIVERY. This is a boundary invented by ShopAssist's taxonomy, not a linguistic fact about the world, so it can only be learned from labeled examples.
- **Calibration for a closed 11-way decision** — pretraining objectives (masked-token or next-token prediction) are not the same objective as "commit to one of these eleven buckets," so the classification head (encoder) or the constrained generation format (decoder) has to learn what confident-vs-uncertain looks like specifically for this task, from scratch.

In short: pretraining supplies the language model, fine-tuning supplies the routing model.

### 6.2 On Handling Chit-Chat Queries

A message like *"Hey! How's it going?"* carries no signal toward any of the 11 categories — it's out-of-distribution for the task, not a hard example of it.

**Encoder approach:** the classification head is a softmax over exactly 11 classes with no reject option. It is mathematically required to output a full probability distribution that sums to 1, so it *will* pick something — most likely whichever category has the vaguest, most generic phrasing in the training data (plausibly CONTACT or FEEDBACK, since those categories' training examples tend to be shorter and less lexically specific than, say, PAYMENT). The failure is **silent**: nothing in the raw output distinguishes this from a real, confident routing decision unless the calling system inspects the softmax margin.

**Decoder approach:** because output is free-form generation rather than a forced softmax, it has more room to fail informatively — it might generate something that doesn't match any category (counted as `UNPARSEABLE` in our evaluation above), or it might still confidently generate a plausible-sounding but wrong category, depending on how the fine-tuning data shaped its behavior on out-of-template input. Empirically here, the SLM's unparseable rate on the actual test set (see Section 3) is the closest proxy we have for "how often does it defensibly refuse," and even that is more a training-format artifact than a designed reject mechanism — it is not reliably more graceful, just capable of a different failure mode.

**Which fails more gracefully:** neither is well-behaved out of the box, but the decoder's generation format at least has a *structural path* to producing a non-answer, where the encoder's forced-softmax design has none. That's a thin advantage, not a solved problem — it depends heavily on what the SLM saw in training, and here it saw the same 11-category-only data as the encoder, with no negative/OOS examples to learn a reject behavior from.

**What I'd add for production:** an explicit **12th class — `OUT_OF_SCOPE`/chit-chat** — trained on non-support messages (greetings, small talk, off-topic chatter) mined or synthetically generated, so the model has actually seen this input at training time instead of extrapolating from support-only data. This would sit alongside (not instead of) the softmax-margin threshold from the Recommendation above: high-confidence `OUT_OF_SCOPE` predictions get a friendly canned response or hand-off to a general-purpose agent, and low-margin predictions across *any* class — including cases where `OUT_OF_SCOPE` and a real category are close — get routed to human review rather than auto-dispatched.